In [1]:
import os
from typing import List, Dict, Any
from getpass import getpass

from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.vectorstores import Chroma
from langchain.llms import HuggingFaceHub
from langchain.chains import RetrievalQA
from langchain.schema import Document

ModuleNotFoundError: No module named 'langchain.document_loaders'

Loading the Web Data

In [ ]:
class WebContentLoader:
    def __init__(self, urls: List[str]):
        self.urls = urls
        
    def load_content(self) -> List[Document]:
        loader = WebBaseLoader(self.urls)
        try:
            documents = loader.load()
            print(f"Successfully loaded content from {len(self.urls)} URLs")
            return documents
        except Exception as e:
            print(f"Error loading content: {str(e)}")
            return []

Chunking the Text

In [ ]:
class DocumentChunker:
    def __init__(self, chunk_size: int = 256, chunk_overlap: int = 50):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
    
    def create_chunks(self, documents: List[Document]) -> List[Document]:
        chunks = self.splitter.split_documents(documents)
        print(f"Created {len(chunks)} chunks from {len(documents)} documents")
        return chunks

Embeddings

class HuggingFaceEmbeddings:
    def __init__(self, model_name: str = "BAAI/bge-base-en-v1.5"):
        # Get HuggingFace token if not already set
        if 'HUGGINGFACEHUB_API_TOKEN' not in os.environ:
            hf_token = getpass("Enter your HuggingFace API token: ")
            os.environ['HUGGINGFACEHUB_API_TOKEN'] = hf_token
            
        self.embeddings = HuggingFaceInferenceAPIEmbeddings(
            api_key=os.environ['HUGGINGFACEHUB_API_TOKEN'],
            model_name=model_name
        )
    
    def get_embeddings(self):
        return self.embeddings

vector Database

class VectorStore:
    def __init__(self, embeddings):
        self.embeddings = embeddings
        self.vectorstore = None
    
    def create_store(self, documents: List[Document]) -> Chroma:
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings
        )
        return self.vectorstore

Retrieval

In [ ]:
class Retriever:
    def __init__(self, vectorstore: Chroma, k: int = 3):
        self.retriever = vectorstore.as_retriever(
            search_type="mmr",  # Maximum Marginal Relevance
            search_kwargs={"k": k}
        )
    
    def get_relevant_documents(self, query: str) -> List[Document]:
        return self.retriever.get_relevant_documents(query)

Prompt Templates

In [ ]:
class PromptManager:
    @staticmethod
    def create_zephyr_prompt(query: str, context: str = "") -> str:
        return f"""
                    <|system|>
                    You are an AI Assistant that follows instructions extremely well.
                    Please be truthful and give direct answers. Please tell 'I don't know' if user query is not in context
                    </s>
                    <|user|>
                    Context: {context}

                    Question: {query}
                    </s>
                    <|assistant|>
                    """

Model Generation

In [ ]:
class ResponseGenerator:
    def __init__(self, model_id: str = "HuggingFaceH4/zephyr-7b-alpha"):
        self.model = HuggingFaceHub(
            repo_id=model_id,
            model_kwargs={
                "temperature": 0.5,
                "max_new_tokens": 512,
                "max_length": 64
            }
        )
        
    def create_qa_chain(self, retriever) -> RetrievalQA:
        return RetrievalQA.from_chain_type(
            llm=self.model,
            retriever=retriever,
            chain_type="stuff"
        )

RAG Pipeline

In [2]:
class RAGPipeline:
    def __init__(self, urls: List[str]):
        self.loader = WebContentLoader(urls)
        self.chunker = DocumentChunker()
        self.embeddings = HuggingFaceEmbeddings()
        self.vectorstore = None
        self.retriever = None
        self.generator = None

    def build(self):
        documents = self.loader.load_content()
        chunks = self.chunker.create_chunks(documents)
        vector_store = VectorStore(self.embeddings.get_embeddings())
        self.vectorstore = vector_store.create_store(chunks)
        retriever_component = Retriever(self.vectorstore)
        self.retriever = retriever_component.retriever
        self.generator = ResponseGenerator()
        self.qa_chain = self.generator.create_qa_chain(self.retriever)
        
    def query(self, question: str) -> str:
        prompt = PromptManager.create_zephyr_prompt(question)
    
        response = self.qa_chain(prompt)
        return response['result']

Example Usage

In [ ]:
def main():
    # Using two of our amazing articles as examples
    urls = [
        "https://www.geeksforgeeks.org/nlp/stock-price-prediction-project-using-tensorflow/",
        "https://www.geeksforgeeks.org/deep-learning/training-of-recurrent-neural-networks-rnn-in-tensorflow/"
    ]
    
    pipeline = RAGPipeline(urls)
    pipeline.build()
    
    query = "What is recurrent neural network?"
    response = pipeline.query(query)
    
    print(f"\nQuery: {query}")
    print(f"Response: {response}")

if __name__ == "__main__":
    main()